# Snowpark Python Workshop: ShopStream Retail Pipeline

**Duration:** 30 minutes | **Level:** Intermediate

## Story
ShopStream is a growing e-commerce company. We'll build a complete Python-based data pipeline on Snowflake using Snowpark:

1. **DataFrames** - Explore and transform retail data
2. **UDFs** - Custom business logic (discount tier calculation)
3. **Stored Procedures** - Production-ready ETL pipeline

---

## Prerequisites
- Run `01_setup.sql` in Snowsight before starting this notebook
- Install: `pip install snowflake-snowpark-python`

## 1. Connect to Snowflake

In [ ]:
from snowflake.snowpark import Session
from snowflake.snowpark.functions import (
    col, lit, sum, avg, count, count_distinct,
    date_trunc, when, upper, udf, pandas_udf, sproc
)
from snowflake.snowpark.types import (
    StringType, FloatType, PandasSeriesType
)

# Create session using connection name from ~/.snowflake/connections.toml
session = Session.builder.config("connection_name", "default").getOrCreate()

print(f"Connected! Role: {session.get_current_role()}")
print(f"Warehouse: {session.get_current_warehouse()}")
print(f"Database: {session.get_current_database()}")

## 2. Loading Data as DataFrames

Key insight: `session.table()` returns a **lazy** DataFrame. No query runs until you call `.show()`, `.collect()`, or `.count()`.

In [ ]:
# Load tables - these are LAZY (no query runs yet!)
customers = session.table("SHOPSTREAM.RAW.CUSTOMERS")
products = session.table("SHOPSTREAM.RAW.PRODUCTS")
orders = session.table("SHOPSTREAM.RAW.ORDERS")

# .show() triggers execution
print(f"Customers: {customers.count()} rows")
print(f"Products: {products.count()} rows")
print(f"Orders: {orders.count()} rows")

customers.show(5)

## 3. Select, Filter, and Transform

Use `col()` to reference columns. Chain operations like `.select()`, `.filter()`, `.with_column()`.

In [ ]:
# Select specific columns
customers.select("FIRST_NAME", "REGION", "SIGNUP_DATE").show(5)

In [ ]:
# Filter: customers in the North region
north_customers = customers.filter(col("REGION") == "North")
print(f"North region customers: {north_customers.count()}")
north_customers.show(5)

In [ ]:
# Add computed columns
products_with_tax = products.with_column(
    "PRICE_WITH_TAX",
    col("UNIT_PRICE") * lit(1.08)
)
products_with_tax.select("PRODUCT_NAME", "UNIT_PRICE", "PRICE_WITH_TAX").show(5)

In [ ]:
# Conditional logic with WHEN
products_tiered = products.select(
    col("PRODUCT_NAME"),
    col("UNIT_PRICE"),
    when(col("UNIT_PRICE") > 200, "Premium")
    .when(col("UNIT_PRICE") > 50, "Standard")
    .otherwise("Budget")
    .alias("PRICE_TIER")
)
products_tiered.show(10)

## 4. Joins

Join orders with customers and products to get a full picture.

In [ ]:
# Build a denormalized order details view
order_details = (
    orders
    .join(customers, orders["CUSTOMER_ID"] == customers["CUSTOMER_ID"])
    .join(products, orders["PRODUCT_ID"] == products["PRODUCT_ID"])
    .select(
        orders["ORDER_ID"],
        customers["FIRST_NAME"],
        customers["REGION"],
        products["PRODUCT_NAME"],
        products["CATEGORY"],
        orders["QUANTITY"],
        (orders["QUANTITY"] * products["UNIT_PRICE"]).alias("ORDER_TOTAL"),
        orders["ORDER_DATE"]
    )
)

order_details.show(10)

## 5. Aggregations

Group by region, month, product - and calculate metrics.

In [ ]:
# Revenue by region
revenue_by_region = (
    order_details
    .group_by("REGION")
    .agg(
        sum("ORDER_TOTAL").alias("TOTAL_REVENUE"),
        count("ORDER_ID").alias("ORDER_COUNT"),
        avg("ORDER_TOTAL").alias("AVG_ORDER_VALUE")
    )
    .sort(col("TOTAL_REVENUE").desc())
)

revenue_by_region.show()

In [ ]:
# Monthly revenue trend
monthly_revenue = (
    order_details
    .group_by(date_trunc("MONTH", col("ORDER_DATE")).alias("MONTH"))
    .agg(
        sum("ORDER_TOTAL").alias("REVENUE"),
        count_distinct("FIRST_NAME").alias("UNIQUE_CUSTOMERS")
    )
    .sort("MONTH")
)

monthly_revenue.show(12)

## 6. User-Defined Functions (UDFs)

Extend Snowflake with custom Python logic. UDFs run **on Snowflake's compute**, not locally.

### Scalar UDF: Discount Tier Calculator

In [ ]:
# Define a scalar UDF
@udf(
    name="discount_tier",
    is_permanent=False,
    replace=True,
    input_types=[FloatType()],
    return_type=StringType()
)
def discount_tier(total_spend: float) -> str:
    """Assign discount tier based on total spend."""
    if total_spend >= 500:
        return "Platinum"
    elif total_spend >= 200:
        return "Gold"
    elif total_spend >= 100:
        return "Silver"
    else:
        return "Bronze"

print("UDF 'discount_tier' registered!")

In [ ]:
# Calculate total spend per customer
customer_spend = (
    orders
    .join(products, orders["PRODUCT_ID"] == products["PRODUCT_ID"])
    .group_by(orders["CUSTOMER_ID"])
    .agg(sum(col("QUANTITY") * col("UNIT_PRICE")).alias("TOTAL_SPEND"))
)

# Apply UDF
customer_tiers = (
    customer_spend
    .join(customers, customer_spend["CUSTOMER_ID"] == customers["CUSTOMER_ID"])
    .select(
        customers["FIRST_NAME"],
        customers["REGION"],
        customer_spend["TOTAL_SPEND"],
        discount_tier(col("TOTAL_SPEND")).alias("DISCOUNT_TIER")
    )
    .sort(col("TOTAL_SPEND").desc())
)

customer_tiers.show(15)

### Vectorized UDF (Pandas UDF)

Processes data in batches using Pandas - much faster for large datasets!

In [ ]:
@pandas_udf(
    name="calculate_discount_amount",
    is_permanent=False,
    replace=True,
    input_types=[PandasSeriesType(FloatType()), PandasSeriesType(StringType())],
    return_type=PandasSeriesType(FloatType())
)
def calculate_discount_amount(spend, tier):
    """Calculate discount amount based on spend and tier."""
    import pandas as pd
    discount_rates = {"Platinum": 0.15, "Gold": 0.10, "Silver": 0.05, "Bronze": 0.00}
    rates = tier.map(discount_rates).fillna(0.0)
    return spend * rates

# Apply vectorized UDF
customer_discounts = customer_tiers.with_column(
    "DISCOUNT_AMOUNT",
    calculate_discount_amount(col("TOTAL_SPEND"), col("DISCOUNT_TIER"))
)

customer_discounts.show(10)

## 7. Stored Procedures

Package the entire pipeline as a callable Stored Procedure.

In [ ]:
def build_monthly_revenue_summary(session: Session) -> str:
    """Full ETL pipeline packaged as a stored procedure."""
    
    # Load
    customers = session.table("SHOPSTREAM.RAW.CUSTOMERS")
    products = session.table("SHOPSTREAM.RAW.PRODUCTS")
    orders = session.table("SHOPSTREAM.RAW.ORDERS")
    
    # Transform
    order_details = (
        orders
        .join(customers, orders["CUSTOMER_ID"] == customers["CUSTOMER_ID"])
        .join(products, orders["PRODUCT_ID"] == products["PRODUCT_ID"])
        .select(
            orders["ORDER_ID"],
            orders["ORDER_DATE"],
            customers["REGION"],
            (orders["QUANTITY"] * products["UNIT_PRICE"]).alias("ORDER_TOTAL")
        )
    )
    
    monthly_summary = (
        order_details
        .group_by(
            col("REGION"),
            date_trunc("MONTH", col("ORDER_DATE")).alias("MONTH")
        )
        .agg(
            sum("ORDER_TOTAL").alias("TOTAL_REVENUE"),
            count("ORDER_ID").alias("ORDER_COUNT"),
            avg("ORDER_TOTAL").alias("AVG_ORDER_VALUE")
        )
        .with_column(
            "REVENUE_TIER",
            when(col("TOTAL_REVENUE") > 5000, "High")
            .when(col("TOTAL_REVENUE") > 2000, "Medium")
            .otherwise("Low")
        )
    )
    
    # Save
    target = "SHOPSTREAM.ANALYTICS.MONTHLY_REVENUE_SUMMARY"
    monthly_summary.write.mode("overwrite").save_as_table(target)
    
    row_count = session.table(target).count()
    return f"SUCCESS: Saved {row_count} rows to {target}"


# Register as Stored Procedure
session.sproc.register(
    func=build_monthly_revenue_summary,
    name="build_monthly_revenue_summary_sp",
    replace=True,
    is_permanent=False
)

print("Stored Procedure registered!")

In [ ]:
# Call the stored procedure
result = session.sql("CALL build_monthly_revenue_summary_sp()").collect()
print(f"Result: {result[0][0]}")

# View the output
session.table("SHOPSTREAM.ANALYTICS.MONTHLY_REVENUE_SUMMARY").sort("MONTH").show(10)

## 8. Summary

| Concept | What You Learned |
|---------|------------------|
| **Session** | Connect to Snowflake from Python |
| **DataFrames** | Lazy evaluation, col(), filter/join/group_by/agg |
| **Scalar UDF** | Custom row-by-row Python logic |
| **Vectorized UDF** | Batch processing with Pandas (faster) |
| **Stored Procedure** | Deploy full pipelines, callable from SQL |

### Key Principles
- Everything runs on **Snowflake's compute** (not your laptop)
- DataFrames are **lazy** - nothing executes until you call `.show()` / `.collect()`
- Use the `main(session)` pattern for code that works **locally AND deployed**
- Prefer native SQL functions over UDFs when possible (performance)
- Use vectorized UDFs over scalar UDFs for large datasets

In [ ]:
# Clean up
session.close()
print("Session closed. Workshop complete!")